In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from goad_toolkit.visualizer import CorrelationHeatmap, PlotSettings, RegPlot, ScatterPlot
from wa_analyzer.data import load_showcase

import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

df = load_showcase("penguins_raw")
select = [
    "Species",
    "Island",
    "Culmen Length (mm)",
    "Culmen Depth (mm)",
    "Flipper Length (mm)",
    "Delta 15 N (o/oo)",
    "Delta 13 C (o/oo)",
    "Sex",
    "Body Mass (g)",
]
subset = df[select].dropna()
subset["species"] = subset["Species"].str.split(" ").str[0]
subset.species.value_counts()

# Correlation

Correlation is a measure of the relationship between two variables. The most common measure of correlation in statistics is the Pearson correlation coefficient, which is a measure of the linear relationship between two variables.

The idea is this: we have been looking at variance, which will compare the difference from every individual datapoint with the mean by calculating $(x_i - \bar{x})^2$ and taking the average of those differences. Now, what if we compare these differences between two variables? That is what the covariance does:

$$cov(x,y) = \frac{1}{n} \sum_i (x_i - \bar{x})(y_i - \bar{y})$$

Now, this is a bit hard to interpret, because it depends on the units of the variables. So, we can normalize this by dividing by the standard deviations of the variables:

$$ r = \frac{cov(x,y)}{\sigma_x \sigma_y}$$

where $\sigma$ is the standard deviation. 


or, if you want to correct for bias:

$$r = \frac{cov(x,y)}{(n-1)\sigma_x \sigma_y}$$

This is the Pearson correlation coefficient. It is a number between -1 and 1, where 1 is a perfect positive correlation, 0 is no correlation, and -1 is a perfect negative correlation.

In [ ]:
floats = subset.select_dtypes(include="float64")
correlation_matrix = floats.corr()
correlation_matrix.round(2)

In [ ]:
settings = PlotSettings(
    figsize=(8, 6),
    title="Six measurements, fifteen correlations",
    xlabel="",
    ylabel="",
)
fig, ax = CorrelationHeatmap(settings).plot(data=floats)

## The variable that is not in the picture

Look at the line that built that matrix:

```python
floats = subset.select_dtypes(include="float64")
```

`Species`, `Island` and `Sex` are text, so they were dropped before a single correlation was
computed. The heatmap cannot show you something it never received — and every one of those
fifteen numbers was measured on three species piled into one heap.

So ask each pair twice: once pooled, and once **within** each species. Subtracting each
species' mean before correlating is the whole of it — it removes the differences *between*
species and leaves the differences *inside* them.

In [ ]:
def within_species(column: str) -> pd.Series:
    """The column with each species' own mean removed."""
    return subset[column] - subset.groupby("species")[column].transform("mean")


pairs = []
columns = list(floats.columns)
for i, a in enumerate(columns):
    for b in columns[i + 1:]:
        pairs.append({
            "pair": f"{a.split(' (')[0]} / {b.split(' (')[0]}",
            "pooled": subset[a].corr(subset[b]),
            "within species": within_species(a).corr(within_species(b)),
        })

pairs = pd.DataFrame(pairs)
pairs["changes sign"] = np.sign(pairs.pooled) != np.sign(pairs["within species"])
print(f"{pairs['changes sign'].sum()} of the {len(pairs)} correlations change sign")
pairs.sort_values("pooled", key=abs, ascending=False).round(2)

**Seven of the fifteen change sign.** One of those is a wobble between two numbers that are
near zero either way; the other six are not — they change direction, so the sentence you
would have written from the heatmap is the opposite of what happens inside every group.

Two of them are worth doing properly. Start with the bills.

In [ ]:
x, y = "Culmen Length (mm)", "Culmen Depth (mm)"

bills = PlotSettings(
    figsize=(12, 4.5),  # ty: ignore[invalid-argument-type]
    title="The same 324 penguins, twice",
    subplot_titles=[f"all of them: r = {subset[x].corr(subset[y]):+.2f}",
                    "one species at a time"],
    xlabel=x,
    ylabel=y,
)

host = ScatterPlot(bills)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(RegPlot(bills), axes[0], data=subset, x=x, y=y, ci=None,
                  scatter_kws={"color": "#999999"})
colours = dict(zip(sorted(subset.species.unique()), ["#4c72b0", "#dd8452", "#55a868"]))
for species, group in subset.groupby("species"):
    host.plot_on_axes(RegPlot(bills), axes[1], data=group, x=x, y=y, ci=None,
                      color=colours[species], label=species,
                      scatter_kws={"alpha": 0.6, "color": colours[species]})
axes[1].legend(title="")

On the left, one downward line through everything: longer bills are shallower, `r = -0.22`.
On the right, three upward lines. Inside every species a longer bill is a *deeper* bill, and
it is not marginal — `+0.40` for Adelie, `+0.65` for Chinstrap, `+0.66` for Gentoo.

Nothing about the birds changed. The pooled line is measuring the fact that Gentoos have long
shallow bills and Adelies have short deep ones, which arranges the three clouds along a
downward diagonal. The slope you see is **between** species; the slope inside them points the
other way.

This is **Simpson's paradox**, and lesson 2 met it as a table of Berkeley admissions. Here it
is a scatter, which makes the mechanism visible: you can see the three clouds it is drawing a
line through.

In [ ]:
for a, b in [("Delta 15 N (o/oo)", "Body Mass (g)"),
             ("Culmen Length (mm)", "Body Mass (g)")]:
    print(f"{a.split(' (')[0]} vs {b.split(' (')[0]}")
    print(f"  pooled          r = {subset[a].corr(subset[b]):+.2f}")
    for species, group in subset.groupby("species"):
        print(f"  {species:10s}      r = {group[a].corr(group[b]):+.2f}   (n={len(group)})")
    print(f"  within species  r = {within_species(a).corr(within_species(b)):+.2f}\n")

The first pair is the stronger lesson, because it does not reverse — it **disappears**.

Nitrogen-15 correlates with body mass at `-0.54` pooled, which is a number you would report.
Within species it is `+0.03`: nothing, in all three groups. The correlation was not weakened
by controlling for species, it was *entirely made of* species. δ15N rises with trophic level,
the three species eat differently and happen to differ in size, and that is the whole
relationship. Knowing a penguin's nitrogen tells you nothing about how heavy it is once you
know what it is.

The second pair is why "distrust correlations" is not the lesson. Bill length against body
mass is `+0.59` pooled and `+0.59` within — untouched, positive in every species. A bigger
bird has a bigger bill, inside groups and across them, and no amount of conditioning makes it
go away.

**Same table, same method, three different outcomes.** That is §5.1's credibility grid doing
its job: the response depends on what you find, and you cannot pick it in advance.

### What to do about it

Three ways to handle a confounder, in increasing order of how much they ask of you:

1. **Report it per group.** Three correlations instead of one. Honest, and often the whole
   answer — "within each species, longer bills are deeper" is a real finding, and it is not
   the one the heatmap offered.
2. **Take the group out first**, which is the `within_species` function above: subtract the
   group mean and correlate what is left. The same subtract-the-boring-part move as lesson 3,
   applied to a category rather than a season.
3. **Put it in a model** alongside the variables you care about, and read the coefficients —
   which is the next section, and the rest of your ML course.

And the part no method fixes: **the confounder has to be in your data**. Species was, and
recovering it took one `groupby`. If the Palmer team had not recorded which bird was which,
every number in that heatmap would still be exactly as wrong, and nothing in the frame would
tell you.

> That is the "4th floor" problem from §5.1 in its general form: *no mechanism* does not mean
> no finding, it means the finding is a variable you have not written down yet. Here it was
> `Species`. In your own project it will be something you did not measure, and the only
> defence is asking — out loud, before you write the sentence — *what else is true of the
> groups I am comparing?*

## Finding correlation through regularization
We will start with an unfortunate dataset. We have 500 datapoints, and 100 features. But there is a lot of noise, and a lot of features arent even correlated to the target!

In real life situations, this might happen more often than you like; you get an abundance of features, and you have no idea what is correlated, and what is just noise.

In [ ]:

datafile = Path("../../data/sim/correlation.csv")
df = pd.read_csv(datafile)
df.shape

You might think, lets start with plotting the correlations. Because we have a 100 features, this is too much for a heatmap, so we will limit ourselves to the correlation between the features and the target.

In [ ]:
correlation_matrix = df.corr()
sns.scatterplot(x=correlation_matrix.index, y=correlation_matrix.target)
plt.xticks([])

The target has obviously a correlation of 1 with itself. But the other features are much more noisy. 6 features have a correlation above 0.2, but that is still not very high.
Maybe the two features with a correlation above 0.5 are worth looking at, but the rest might be noise if you use this method. Because this is synthetic data that I created myself, I know for a fact that 10 out of 100 features are relevant. So, for this case plain correlation is not very good at finding the relevant features.

However, there are smarter ways of figuring out a correlation. First, let's split the data into features X and target y

In [ ]:
X = df.drop(columns=["target"]).values
y = df["target"].values

X.shape, y.shape

## (mis)using a linear regression for feature selection
What we will do, is we will construct a linear regression for this problem. 

Note that in this strategy, our goal is NOT to create a linear regression model. Well, we might want to do that, but that is not the point here. We are trying to get a better grip on correlation, and which features contain relevant information. We might already have decided we are going to use a random forest, or a neural network, or whatever fits our goal, or we might not have decided yet. In using a linear regression model, we are going to use regularization, and we are going to read out the weights of the model.

These weights is what we are actually interested in, because it will tell us something about how informative the models are. If a weight is zero, it means that the feature is not used in the model. If a weight is high, it means that the feature is used a lot in the model. If a weight is negative, it means that the feature is used inversely in the model.

This means we are going to assume there is a relation like this:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_{100} x_{100}$$

Where $\beta$ are the weights we want to learn.

We are going to define a "loss" function. This function tells us how good or bad our model is working. The better the model, the lower the loss. The worse the model, the higher the loss. The mean squared error is a common loss function, that calculates the average of the squared differences between the predictions and the actual values:

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2$$

# Using regularization
But a 100 features is a lot, especially compared to 500 data points. So we are going to restrain the weights by adding a penalty to the loss function. This is called regularization. 

## L2 regularization
Ridge regression (often called "l2") looks like this:

$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + \alpha \frac{1}{2} \sum_j \beta_j^2$


The first part is the normal loss function, the second part $\alpha \frac{1}{2} \sum_j \beta_j^2$ is the regularization.
The $\frac{1}{2}$ helps with taking the gradient of the function: as you might remember from school when you were 16, the derivative of $a^2$ is $2a$, but the derivartive of 
$\frac{1}{2}a^2$ is $\frac{1}{2} 2 a$, which is simplified to just $a$ which is convenient.

What we actually do is adding the square of every weight as a penalty.

Can you understand what this does? The weights are an additional term that we try to minimize.
If two models are almost equally good, but one of them has lower weights, this model will be preferred.
$\alpha$ is a hyperparameter that controls the strength of the regularization. The higher $\alpha$, the more the weights will be restrained.


## L1 regularization
Another way of regularization is Lasso (often called "l1"):

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|$$

You notice the same mean square error loss function $\frac{1}{n} \sum_i (y_i - \hat{y}_i)^2$, but now we are adding the summed ($\Sigma$) absolute value (the $||$ mean absolute value. $|x|$ is always positive, even if $x$ is negative) of the weights $\beta$ as a penalty, where $\alpha$ is a parameter that determines if this part has a lot of impact (when $\alpha$ is big) or almost no impact (when $\alpha$ is small): $\alpha \sum_j |\beta_j|$

## Elasticnet
The difference between l1 and l2 is that l1 will tend to generate more sparse weights. That is why l1 if often used for feature selection. Another option is to simply mix the two strategies, and search for a parameter that balances the two. This combination is called ElasticNet.

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + r\alpha \sum_j |\beta_j| + \alpha \frac{1-r}{2} \sum_j \beta_j^2$$

This formula looks impressive, but lets break it down:

- the first part is the usual loss function $\frac{1}{n} \sum_i (y_i - \hat{y}_i)^2$
- the second part is the l1 regularization $\alpha \sum_j |\beta_j|$. If $r$ is zero, this is removed from the equation.
- the third part is the l2 regularization $\alpha \frac{1-r}{2} \sum_j \beta_j^2$. If $r$ is one, this is removed from the equation. 

Now, we can pick a value between 0 and 1 for r, and this will balance the two regularization strategies.

First, let's use it with default values

In [ ]:
from sklearn.linear_model import SGDRegressor

regressor = SGDRegressor(penalty="elasticnet", random_state=42)
regressor.fit(X, y)
coef = regressor.coef_
plt.plot(coef.T, "o")  # ty: ignore[unresolved-attribute] -- coef_ is set once fit() has run

I think this is an impressive improvement over using simple correlation!

We have the option to try different values for $r$ and $\alpha$. While creating models and hypertuning is a topic for the next semester, I will show it here so you already have a look at it.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV


l1_ratio = [0.1, 0.15, 0.3, 0.6, 0.75, 0.9, 0.95, 0.99, 1]
alphalist = [0.001, 0.01, 0.1, 1, 10, 100]

regressor = SGDRegressor(penalty="elasticnet", random_state=42)
param_grid = {"alpha":alphalist, "l1_ratio":l1_ratio}

grid = GridSearchCV(
    regressor,
    param_grid=param_grid,
    cv=5,
)


grid.fit(X, y)
best_model = grid.best_estimator_
coef = best_model.coef_
plt.plot(coef.T, "o")

In [ ]:
grid.best_params_

If we look at the result, you can see that the grid picked `l1_ratio = 1`, which means that the search found the L1 regularisation to be the best one for this problem.
